# PS2 overlap diagnostic (v4)

Read-only. Does not write to S3, does not touch the Gold layer, does not publish, and does
not modify the PS2 notebook or any PS2 output.

**Why this exists.** The episode-closure diagnostic disproved the theory that OOS episodes
never close: `clear_per_set` was ~1.00 on every fleet, `pct_unclosed` was 0.00, and
`device_outage` had zero rows with a null `outage_end`. Boundary censoring at 11 April is
negligible. But the recorded totals still imply **22.9 out-of-service hours per validator
per day**, sustained over 210 days, and a 24-hour day cannot contain that.

One candidate remains. PS2 computes `hardware_oos_minutes` as `sum(device_outage.duration_min)`.
If a device can be in several OOS episodes at once -- different components failing together,
which is precisely what PS2's cascade thesis claims -- then that sum counts the same
wall-clock minute once per concurrent episode. This notebook measures that rather than
assuming it.

**What changed in v4, and why it matters more than the earlier fixes.** The v3 run
completed and gave a number, but the number was not a valid comparison. `overlap_factor`
divides `sum(duration_min)` by the union of the intervals -- and v3 built those intervals
from `outage_end - outage_start`, while the sum uses `duration_min`. A.1 showed those are
**not the same quantity**: `duration_min` is the span clipped at 7 days, and 20,991 rows
carry a year-3000 `outage_end` that contributes up to 210 clamped days to the union but
only 7 to the sum. The proof that this mattered is TVM, which came back with
`overlap_factor = 0.905` -- a union *larger* than its own sum, which is arithmetically
impossible for one interval set.

v4 builds the interval as **`[start, start + duration_min]`**, clamped to the window. That
is sentinel-free by construction and is the exact quantity PS2 sums, so the two sides of
the ratio finally measure the same thing. The raw-span union is still reported beside it as
`union_minutes_span_basis` so the contamination stays visible instead of being quietly
dropped. Verified: across 400 randomised cases with sentinel ends injected, the union never
exceeds the sum under the new basis -- the invariant v3 violated.

**What it does.**

- **A** Does `duration_min` even agree with `outage_end - outage_start`? Plus the 7-day
  duration cap, exact-duplicate intervals, and -- new in v3 -- what actually distinguishes
  the rows that share one interval. The first run found that **55.4% of TVM outage rows**
  share an exact (device, start, end) with another row, one interval carrying **189 rows**.
  `source_event_id` was already shown to be unique across them, so `A.3` asks whether the
  subsystem, serial or failure level differs, or whether they are indistinguishable.
- **B** The decisive test: merge overlapping intervals per device and compare the union
  against the sum. `overlap_factor` is how many times the average out-of-service minute is
  being counted.
- **C** How deep concurrency goes, and what *kind* it is -- different subsystems (legitimate
  multi-component recording, fixable in the notebook) or repeated event ids (a source
  defect, which is a conversation with Cubic instead).
- **E** The restated share of available device-hours, computed from the union and clipped
  per calendar day so no device can exceed 1440 minutes in a day by construction. The
  ceiling check must print 0.

**What changed in v3.** The first attempt crashed the Spark driver. Two causes, both mine:

- The per-day split ran `explode(sequence(...))` on raw timestamps. Rows were filtered on
  `outage_start` inside the window but `outage_end` was never bounded, and this source is
  known to carry sentinel dates near the year 3000 -- one such row generates roughly
  356,000 exploded rows on its own. Intervals are now **clamped to the analysis window
  before anything else**, which is also more correct: only downtime inside the window
  should count. `A.0` reports how many rows ran past the boundary rather than hiding it,
  and cell 3 asserts the bound holds before exploding.
- The block-composition test joined episodes back to blocks on a range condition, which for
  a validator with ~2,000 episodes approaches a per-device cross product. The merged block
  id is now carried on the original rows, so there is no join at all.
- `A.0` collected `max(outage_end)` as a timestamp. That column holds year-3000 sentinels
  and pandas `timestamp[ns]` cannot represent anything past 2262-04-11, so the collection
  raised `ArrowInvalid`. It is now formatted to text in Spark before collection. Every
  other timestamp reaching pandas in this notebook comes from the clamped frame and is
  bounded by the window.

**Verification.** The interval-merge and sweep-line logic were checked against a
brute-force minute-set union over 300 randomised cases plus the awkward shapes real data
contains -- duplicate, nested, touching and zero-length intervals. Zero mismatches. On the
edge-case fixture the union is 30 minutes where the naive sum is 45, which is exactly the
double-counting this run is built to expose.

The clamping was verified separately over 400 further randomised cases with 619 sentinel
end dates injected: the clamped merge reproduces the brute-force in-window union exactly,
and every merged block comes out bounded by the window length.

**To run.** Set `PS2_NOTEBOOK_PATH` if your repo clone is not at
`~/Chicago-Ventra-Mars-Cubic-Analysis`, then Run All. Only PS2 cells 0-4 are executed.
Cell 2 is the heavy one; cells 2 and 3 cache their intermediates so cell 3 does not
recompute the merge.

**What it still does not test.** Whether the underlying episodes are real at all. A union
of spurious intervals is still spurious.


In [ ]:
# =============================================================================
# PS2 OVERLAP DIAGNOSTIC -- CELL 1 of 3: borrow the PS2 setup, change nothing
#
# READ-ONLY. Never writes to S3, never touches Gold, never publishes, never
# modifies the PS2 notebook. Same discipline as the episode-closure diagnostic:
# it execs PS2 code cells 0..4, which are pure setup, so the source paths, the
# fleet mapping and the column contracts cannot drift from the pipeline being
# explained. PS2 cell 5 onward is deliberately NOT run.
#
# WHY THIS RUN EXISTS. The closure diagnostic disproved the unclosed-episode
# theory: clear_per_set was ~1.00 on every fleet, pct_unclosed was 0.00, and
# device_outage had zero null_end rows. But the recorded totals still imply
# 22.9 out-of-service hours per validator per day, which a 24-hour day cannot
# contain. The remaining candidate is that PS2 sums duration_min across
# episodes that overlap in time on the same device, so the same wall-clock
# minute is counted once per concurrent episode. This notebook measures that
# directly instead of assuming it.
# =============================================================================
import os, json
from pathlib import Path

os.environ['PS2_RUN_MODE'] = 'REPLAY'
os.environ['PS2_EXPORT_PREFIX'] = 'ps2_diagnostic_overlap_outputs'
os.environ.setdefault('PS2_MAX_SOURCE_LAG_DAYS', '150')

PS2_NOTEBOOK = Path(os.getenv(
    'PS2_NOTEBOOK_PATH',
    Path.home() / 'Chicago-Ventra-Mars-Cubic-Analysis' / 'notebooks'
    / 'ps2_cascading_failure' / 'PS2_Failure_Patterns_v2_5_3_Storage_Safe.ipynb',
))
if not PS2_NOTEBOOK.exists():
    raise FileNotFoundError(
        f'Cannot find the PS2 notebook at {PS2_NOTEBOOK}. Set PS2_NOTEBOOK_PATH to its '
        'location in the synced repo clone.'
    )

_nb = json.loads(PS2_NOTEBOOK.read_text(encoding='utf-8'))
_code = [''.join(c['source']) for c in _nb['cells'] if c['cell_type'] == 'code']
SETUP_CELLS = 5
if len(_code) < SETUP_CELLS:
    raise RuntimeError(f'Expected at least {SETUP_CELLS} code cells in PS2, found {len(_code)}.')

_forbidden = ('put_object', 'upload_file', 'saveAsTable', '.write.', 'delete_object')
for _i in range(SETUP_CELLS):
    _hit = [t for t in _forbidden if t in _code[_i]]
    if _hit:
        raise RuntimeError(
            f'PS2 setup cell {_i} now contains a write operation {_hit}; this diagnostic '
            'will not exec it. Review the notebook before continuing.'
        )

for _i in range(SETUP_CELLS):
    print(f'--- exec PS2 code cell {_i} ({len(_code[_i]):,} chars)')
    exec(compile(_code[_i], f'<PS2 code cell {_i}>', 'exec'), globals())

for _needed in ('spark', 'read_source', 'canonical_device_category', 'truthy',
                'TARGET_DEVICE_CATEGORIES', 'ANALYSIS_LOOKBACK_DAYS', 'SOURCE'):
    if _needed not in globals():
        raise RuntimeError(f'PS2 setup did not provide {_needed!r}; cannot continue.')

print()
print('Setup borrowed from PS2. No PS2 load or publish cell was run.')
print('outage ->', SOURCE['outage']['path'])


In [ ]:
# =============================================================================
# PS2 OVERLAP DIAGNOSTIC -- CELL 2 of 3: do episodes overlap, and by how much?
#
# PS2 computes hardware_oos_minutes as sum(device_outage.duration_min). If a
# device can be in several OOS episodes at once -- different components failing
# together, which is what PS2's cascade thesis says happens -- then that sum
# counts the same wall-clock minute once per concurrent episode. The honest
# quantity is the UNION of the intervals per device.
#
#   A  does duration_min agree with (outage_end - outage_start)?
#   B  merged union minutes vs summed minutes, per fleet
#
# v3 CHANGE: intervals are CLAMPED to the analysis window before anything else.
# The previous run crashed the driver because outage_end is not bounded by the
# window filter, and this source is already known to contain sentinel dates
# around the year 3000 -- one of those makes a later per-day explode generate
# ~356,000 rows by itself. Clamping is also more correct: we are measuring
# downtime INSIDE the window, so an interval that runs past the boundary should
# be counted only up to it. How many rows were clamped is reported, not hidden.
#
# The merged block id is kept on the original rows so cell 3 never has to join
# episodes back to blocks on a range condition.
#
# READ-ONLY.
# =============================================================================
import datetime as dt
import pandas as pd
from pyspark.sql import functions as F, Window
from pyspark import StorageLevel

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)

TARGETS = list(TARGET_DEVICE_CATEGORIES)
DAY = 1440.0

_ev = (
    read_source('label_events',
                ['DEVICE_ID', 'EVENT_DTM', 'EVENT_STATE_TYPE_NAME', 'is_hardware_oos_event',
                 'mars_device_category'], [])
    .withColumn('fleet', canonical_device_category(F.col('mars_device_category')))
    .where(F.col('fleet').isin(*TARGETS))
    .where(truthy(F.col('is_hardware_oos_event')))
)
WIN_END = _ev.agg(F.max(F.col('EVENT_DTM').cast('timestamp')).alias('m')).first()['m']
WIN_START = WIN_END - dt.timedelta(days=ANALYSIS_LOOKBACK_DAYS)
WIN_DAYS = ANALYSIS_LOOKBACK_DAYS
print(f'Window: {WIN_START} -> {WIN_END}  ({WIN_DAYS} days)')
print()

og_raw = read_source(
    'outage',
    ['DEVICE_ID', 'outage_start', 'outage_end', 'duration_min', 'mars_device_category'],
    ['source_event_id', 'is_chargeable', 'failure_level', 'component_subsystem',
     'COMPONENT_SERIAL_NBR'],
)
print('columns projected from device_outage:', og_raw.columns)
print()

og_raw = (
    og_raw
    .withColumn('fleet', canonical_device_category(F.col('mars_device_category')))
    .withColumn('device_id', F.col('DEVICE_ID').cast('string'))
    .withColumn('raw_start', F.col('outage_start').cast('timestamp'))
    .withColumn('raw_end', F.col('outage_end').cast('timestamp'))
    .withColumn('dur_min', F.col('duration_min').cast('double'))
    .where(F.col('fleet').isin(*TARGETS))
    .where((F.col('raw_start') >= F.lit(WIN_START)) & (F.col('raw_start') <= F.lit(WIN_END)))
    .where(F.col('device_id').isNotNull() & F.col('raw_start').isNotNull() & F.col('raw_end').isNotNull())
    .persist(StorageLevel.DISK_ONLY)
)
n_raw = og_raw.count()

# A.0 -- how far past the window do intervals run, and are there sentinels?
print('A.0  Unbounded outage_end values (this is what crashed the previous run)')
tail = (
    og_raw.groupBy('fleet').agg(
        F.count('*').alias('rows'),
        F.sum(F.when(F.col('raw_end') > F.lit(WIN_END), 1).otherwise(0)).alias('end_after_window'),
        F.sum(F.when(F.col('raw_end') > F.lit(WIN_END) + F.expr('INTERVAL 365 DAYS'), 1).otherwise(0)).alias('end_over_1_year_past'),
        # pandas timestamp[ns] tops out at 2262-04-11 and this column holds
        # year-3000 sentinels, so it is collected as text, never as a timestamp.
        F.date_format(F.max('raw_end'), 'yyyy-MM-dd HH:mm:ss').alias('max_outage_end'),
        F.expr('percentile_approx((unix_timestamp(raw_end) - unix_timestamp(raw_start)) / 86400.0, 0.999)').alias('p999_span_days'),
        F.max((F.col('raw_end').cast('long') - F.col('raw_start').cast('long')) / 86400.0).alias('max_span_days'),
    ).toPandas()
)
print(tail.sort_values('fleet').to_string(index=False))
print()

# v4 CHANGE, and it is the one that makes B.1 mean anything.
#
# The quantity PS2 sums is duration_min. A.1 showed duration_min IS the span for
# 99.5%+ of rows, and for the rest it is the span clipped at 7 days -- exactly
# the rows whose raw outage_end is a year-3000 sentinel. So duration_min is both
# sentinel-free and the precise basis of the metric under investigation.
#
# v3 built the union from (outage_end - outage_start) clamped to the window,
# while the sum used duration_min. Those are different quantities, which is why
# TVM came back with a union LARGER than its own sum -- arithmetically
# impossible for one interval set, and the proof that the two sides disagreed.
#
# The interval is now [start, start + duration_min], clamped to the window. The
# raw span is retained as end_span_ts so the two can still be contrasted.
_end_dur = F.to_timestamp(F.from_unixtime(
    F.col('raw_start').cast('long') + (F.col('dur_min') * 60.0).cast('long')))
og = (
    og_raw
    .withColumn('start_ts', F.greatest('raw_start', F.lit(WIN_START)))
    .withColumn('end_ts', F.least(F.greatest(_end_dur, F.col('raw_start')), F.lit(WIN_END)))
    .withColumn('end_span_ts', F.least('raw_end', F.lit(WIN_END)))
    .where(F.col('end_ts') >= F.col('start_ts'))
    .withColumn('span_min', (F.col('end_span_ts').cast('long') - F.col('start_ts').cast('long')) / 60.0)
    .withColumn('interval_min', (F.col('end_ts').cast('long') - F.col('start_ts').cast('long')) / 60.0)
    .persist(StorageLevel.DISK_ONLY)
)
n_rows = og.count()
print(f'device_outage rows in window: {n_rows:,} (of {n_raw:,}; '
      f'{n_raw - n_rows:,} dropped because the clamped interval inverted)')
print()

# ---------------------------------------------------------------- A: duration vs span
print('A.1  Does duration_min agree with (outage_end - outage_start), after clamping?')
agree = (
    og.groupBy('fleet').agg(
        F.count('*').alias('episodes'),
        F.sum(F.when(F.abs(F.col('dur_min') - F.col('span_min')) <= 0.5, 1).otherwise(0)).alias('agree_within_30s'),
        F.sum(F.when(F.col('dur_min') > F.col('span_min') + 0.5, 1).otherwise(0)).alias('duration_gt_span'),
        F.sum(F.when(F.col('dur_min') < F.col('span_min') - 0.5, 1).otherwise(0)).alias('duration_lt_span'),
        F.expr('percentile_approx(span_min - dur_min, 0.5)').alias('p50_span_minus_duration'),
        F.max(F.col('span_min') - F.col('dur_min')).alias('max_span_minus_duration'),
        F.sum('dur_min').alias('sum_duration_min'),
        F.sum('span_min').alias('sum_span_min'),
    ).toPandas()
)
agree['pct_agree'] = (100.0 * agree['agree_within_30s'] / agree['episodes']).round(2)
print(agree.sort_values('fleet').to_string(index=False))
print()

print('A.2  The 7-day cap, and exact-duplicate intervals')
dup = (
    og.groupBy('fleet', 'device_id', 'start_ts', 'end_ts').agg(F.count('*').alias('n'))
    .groupBy('fleet').agg(
        F.count('*').alias('distinct_intervals'),
        F.sum('n').alias('rows'),
        F.sum(F.when(F.col('n') > 1, F.col('n') - 1).otherwise(0)).alias('duplicate_rows'),
        F.max('n').alias('max_rows_on_one_interval'),
    ).toPandas()
)
cap = (
    og.groupBy('fleet').agg(
        F.sum(F.when(F.col('dur_min') >= 10080.0, 1).otherwise(0)).alias('at_7_day_cap'),
        F.sum(F.when(F.col('span_min') > 10080.0, 1).otherwise(0)).alias('span_over_7_days'),
        F.sum(F.when(F.col('dur_min') == 0.0, 1).otherwise(0)).alias('zero_duration'),
    ).toPandas()
)
capdup = dup.merge(cap, on='fleet')
capdup['pct_duplicate_rows'] = (100.0 * capdup['duplicate_rows'] / capdup['rows']).round(2)
print(capdup.sort_values('fleet').to_string(index=False))
print()

# ---------------------------------------------------------------- A.3: what ARE the duplicates
# v3 ADDITION, prompted by the first run: 55.4% of TVM outage rows share an exact
# (device, start, end) with another row, one interval carrying up to 189 rows.
# Whether that is 189 genuinely distinct component records that happen to share
# timing, or the same episode written 189 times, decides whether the fix is ours
# or Cubic's. source_event_id was already shown to be unique, so the question is
# what else differs.
print('A.3  Exact-duplicate intervals: what distinguishes the rows sharing one interval?')
def _o(name, alias):
    return (F.coalesce(F.col(name).cast('string'), F.lit('(null)')).alias(alias)
            if name in og.columns else F.lit('(not projected)').alias(alias))

dup_detail = (
    og.select('fleet', 'device_id', 'start_ts', 'end_ts',
              _o('component_subsystem', 'subsystem'),
              _o('COMPONENT_SERIAL_NBR', 'serial'),
              _o('source_event_id', 'event_id'),
              _o('failure_level', 'failure_level'))
    .groupBy('fleet', 'device_id', 'start_ts', 'end_ts')
    .agg(F.count('*').alias('rows_on_interval'),
         F.countDistinct('subsystem').alias('n_subsystems'),
         F.countDistinct('serial').alias('n_serials'),
         F.countDistinct('event_id').alias('n_event_ids'),
         F.countDistinct('failure_level').alias('n_failure_levels'))
    .where(F.col('rows_on_interval') > 1)
    .persist(StorageLevel.DISK_ONLY)
)
dup_profile = (
    dup_detail.groupBy('fleet').agg(
        F.count('*').alias('duplicated_intervals'),
        F.sum('rows_on_interval').alias('rows_involved'),
        F.sum(F.when(F.col('n_subsystems') > 1, 1).otherwise(0)).alias('differ_by_subsystem'),
        F.sum(F.when(F.col('n_serials') > 1, 1).otherwise(0)).alias('differ_by_serial'),
        F.sum(F.when(F.col('n_failure_levels') > 1, 1).otherwise(0)).alias('differ_by_failure_level'),
        F.sum(F.when((F.col('n_subsystems') == 1) & (F.col('n_serials') == 1)
                     & (F.col('n_failure_levels') == 1), 1).otherwise(0)).alias('indistinguishable_apart_from_event_id'),
        F.max('rows_on_interval').alias('max_rows_on_one_interval'),
    ).toPandas()
)
for column in ('differ_by_subsystem', 'differ_by_serial', 'differ_by_failure_level',
               'indistinguishable_apart_from_event_id'):
    dup_profile['pct_' + column] = (
        100.0 * dup_profile[column] / dup_profile['duplicated_intervals'].clip(lower=1)).round(2)
print(dup_profile.sort_values('fleet').to_string(index=False))
print()
print('A.3b  The ten worst-duplicated intervals')
print(dup_detail.orderBy(F.desc('rows_on_interval')).limit(10).toPandas().to_string(index=False))
print()
dup_detail.unpersist()

# ---------------------------------------------------------------- B: the union
print('B.1  Merging overlapping intervals per device (the decisive test)')
w_dev = Window.partitionBy('fleet', 'device_id').orderBy('start_ts', 'end_ts')
# The block id is attached to the ORIGINAL rows and kept. Cell 3 then derives
# both the block composition and the per-day split from this one frame, with no
# range join anywhere.
og_blocked = (
    og.select('fleet', 'device_id', 'start_ts', 'end_ts', 'dur_min', 'span_min',
              *[c for c in ('component_subsystem', 'COMPONENT_SERIAL_NBR', 'source_event_id') if c in og.columns])
    .withColumn('_prior_max_end', F.max('end_ts').over(w_dev.rowsBetween(Window.unboundedPreceding, -1)))
    .withColumn('_is_new_block',
                F.when(F.col('_prior_max_end').isNull(), F.lit(1))
                 .when(F.col('start_ts') > F.col('_prior_max_end'), F.lit(1))
                 .otherwise(F.lit(0)))
    .withColumn('_block', F.sum('_is_new_block').over(w_dev.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
    .drop('_prior_max_end', '_is_new_block')
    .persist(StorageLevel.DISK_ONLY)
)
og_blocked.count()

merged = (
    og_blocked.groupBy('fleet', 'device_id', '_block')
    .agg(F.min('start_ts').alias('block_start'),
         F.max('end_ts').alias('block_end'),
         F.count('*').alias('episodes_in_block'),
         F.sum('dur_min').alias('summed_duration_min'))
    .withColumn('block_minutes', (F.col('block_end').cast('long') - F.col('block_start').cast('long')) / 60.0)
    .persist(StorageLevel.DISK_ONLY)
)
n_blocks = merged.count()
print(f'merged downtime blocks: {n_blocks:,} (from {n_rows:,} episodes)')
print()

union_by_fleet = (
    merged.groupBy('fleet').agg(
        F.countDistinct('device_id').alias('devices'),
        F.count('*').alias('merged_blocks'),
        F.sum('block_minutes').alias('union_minutes'),
        F.sum('summed_duration_min').alias('summed_minutes'),
        F.max('episodes_in_block').alias('max_episodes_in_one_block'),
        F.expr('percentile_approx(episodes_in_block, 0.5)').alias('p50_episodes_per_block'),
        F.expr('percentile_approx(episodes_in_block, 0.99)').alias('p99_episodes_per_block'),
        F.max('block_minutes').alias('longest_block_minutes'),
    ).toPandas()
)
# Contrast basis: the union built from the RAW span instead of duration_min.
# It is reported so the sentinel contamination is visible rather than implied.
w_span = Window.partitionBy('fleet', 'device_id').orderBy('start_ts', 'end_span_ts')
span_union = (
    og.select('fleet', 'device_id', 'start_ts', 'end_span_ts')
    .withColumn('_pm', F.max('end_span_ts').over(w_span.rowsBetween(Window.unboundedPreceding, -1)))
    .withColumn('_new', F.when(F.col('_pm').isNull(), F.lit(1))
                         .when(F.col('start_ts') > F.col('_pm'), F.lit(1)).otherwise(F.lit(0)))
    .withColumn('_b', F.sum('_new').over(w_span.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
    .groupBy('fleet', 'device_id', '_b')
    .agg(F.min('start_ts').alias('bs'), F.max('end_span_ts').alias('be'))
    .withColumn('m', (F.col('be').cast('long') - F.col('bs').cast('long')) / 60.0)
    .groupBy('fleet').agg(F.sum('m').alias('union_minutes_span_basis')).toPandas()
)
union_by_fleet = union_by_fleet.merge(span_union, on='fleet', how='left')
union_by_fleet['overlap_factor'] = (union_by_fleet['summed_minutes'] / union_by_fleet['union_minutes']).round(3)
union_by_fleet['minutes_double_counted'] = union_by_fleet['summed_minutes'] - union_by_fleet['union_minutes']
union_by_fleet['pct_of_sum_that_is_double_counted'] = (
    100.0 * union_by_fleet['minutes_double_counted'] / union_by_fleet['summed_minutes']).round(2)
print(union_by_fleet.sort_values('fleet').to_string(index=False))
print()
og_raw.unpersist()


In [ ]:
# =============================================================================
# PS2 OVERLAP DIAGNOSTIC -- CELL 3 of 3: concurrency depth, and the restated metric
#
# C  How deep does concurrency go, and are the simultaneous episodes genuinely
#    different components (legitimate multi-component recording) or the same
#    thing recorded repeatedly (a source defect)? Different fixes.
# E  The share of available device-hours recomputed from the union, clipped per
#    calendar day so no device can exceed 1440 minutes in a day.
#
# v3 CHANGES, both of which were driver-killers in the previous run:
#   - composition is read off the block id already carried on og_blocked. The
#     previous version joined episodes back to blocks on a range condition,
#     which for a validator with ~2,000 episodes approaches a per-device cross
#     product.
#   - the per-day split now runs on intervals already clamped to the window in
#     cell 2, so sequence() can emit at most WIN_DAYS+1 rows per block instead
#     of ~356,000 for a single sentinel outage_end.
#
# READ-ONLY.
# =============================================================================
from pyspark.sql import functions as F, Window
from pyspark import StorageLevel
import pandas as pd

# ---------------------------------------------------------------- C: concurrency depth
print('C.1  How many OOS episodes are open at once on the same device?')
edges = (
    og_blocked.select('fleet', 'device_id', F.col('start_ts').alias('ts'), F.lit(1).alias('delta'))
    .unionByName(og_blocked.select('fleet', 'device_id', F.col('end_ts').alias('ts'), F.lit(-1).alias('delta')))
)
w_sweep = (Window.partitionBy('fleet', 'device_id').orderBy('ts', 'delta')
           .rowsBetween(Window.unboundedPreceding, Window.currentRow))
depth_by_device = (
    edges.withColumn('open_now', F.sum('delta').over(w_sweep))
    .groupBy('fleet', 'device_id').agg(F.max('open_now').alias('max_concurrent'))
)
depth_summary = (
    depth_by_device.groupBy('fleet').agg(
        F.count('*').alias('devices'),
        F.sum(F.when(F.col('max_concurrent') <= 1, 1).otherwise(0)).alias('devices_never_concurrent'),
        F.expr('percentile_approx(max_concurrent, 0.5)').alias('p50_max_concurrent'),
        F.expr('percentile_approx(max_concurrent, 0.9)').alias('p90_max_concurrent'),
        F.max('max_concurrent').alias('worst_max_concurrent'),
    ).toPandas()
)
depth_summary['pct_devices_with_concurrency'] = (
    100.0 * (depth_summary['devices'] - depth_summary['devices_never_concurrent'])
    / depth_summary['devices']).round(2)
print(depth_summary.sort_values('fleet').to_string(index=False))
print()

# ---------------------------------------------------------------- C.2: composition
print('C.2  Within a merged block, are the concurrent episodes different components?')
def _opt(name):
    # read_source projects an optional column only when the source has it.
    return (F.coalesce(F.col(name).cast('string'), F.lit('(null)')) if name in og_blocked.columns
            else F.lit('(not projected)'))
print('optional columns available:',
      {c: (c in og_blocked.columns) for c in ('component_subsystem', 'COMPONENT_SERIAL_NBR', 'source_event_id')})

composition = (
    og_blocked
    .select('fleet', 'device_id', '_block',
            _opt('component_subsystem').alias('subsystem'),
            _opt('COMPONENT_SERIAL_NBR').alias('serial'),
            _opt('source_event_id').alias('event_id'))
    .groupBy('fleet', 'device_id', '_block').agg(
        F.count('*').alias('episodes'),
        F.countDistinct('subsystem').alias('distinct_subsystems'),
        F.countDistinct('serial').alias('distinct_serials'),
        F.countDistinct('event_id').alias('distinct_event_ids'),
    )
    .where(F.col('episodes') > 1)
    .groupBy('fleet').agg(
        F.count('*').alias('multi_episode_blocks'),
        F.sum(F.when(F.col('distinct_subsystems') > 1, 1).otherwise(0)).alias('blocks_multi_subsystem'),
        F.sum(F.when(F.col('distinct_serials') > 1, 1).otherwise(0)).alias('blocks_multi_serial'),
        F.sum(F.when(F.col('distinct_event_ids') < F.col('episodes'), 1).otherwise(0)).alias('blocks_with_repeated_event_id'),
        F.expr('percentile_approx(episodes, 0.5)').alias('p50_episodes'),
        F.expr('percentile_approx(distinct_subsystems, 0.5)').alias('p50_distinct_subsystems'),
    ).toPandas()
)
composition['pct_blocks_multi_subsystem'] = (
    100.0 * composition['blocks_multi_subsystem'] / composition['multi_episode_blocks'].clip(lower=1)).round(2)
composition['pct_blocks_repeated_event_id'] = (
    100.0 * composition['blocks_with_repeated_event_id'] / composition['multi_episode_blocks'].clip(lower=1)).round(2)
print(composition.sort_values('fleet').to_string(index=False))
print()

# ---------------------------------------------------------------- E: restated metric
print('E.1  Restated share of available device-hours, from the union, clipped per day')
print('     (intervals are [start, start + duration_min], the exact quantity PS2 sums)')
# Every block was clamped to the window in cell 2, so this sequence is bounded
# by WIN_DAYS+1 elements. Asserted rather than assumed.
_max_block_days = merged.agg(
    F.max((F.col('block_end').cast('long') - F.col('block_start').cast('long')) / 86400.0)).first()[0]
print(f'longest merged block: {_max_block_days:,.2f} days (window is {WIN_DAYS} days)')
if _max_block_days is not None and _max_block_days > WIN_DAYS + 1:
    raise RuntimeError(
        f'A merged block spans {_max_block_days:,.1f} days, longer than the window. The clamp in '
        'cell 2 did not hold; stopping before the per-day split can exhaust the driver.')

exploded = (
    merged.select('fleet', 'device_id', 'block_start', 'block_end')
    .withColumn('day', F.explode(F.sequence(F.to_date('block_start'), F.to_date('block_end'))))
    .withColumn('day_start', F.col('day').cast('timestamp'))
    .withColumn('day_end', F.col('day_start') + F.expr('INTERVAL 1 DAY'))
    .withColumn('ov_start', F.greatest('block_start', 'day_start'))
    .withColumn('ov_end', F.least('block_end', 'day_end'))
    .withColumn('minutes_in_day',
                F.greatest(F.lit(0.0), (F.col('ov_end').cast('long') - F.col('ov_start').cast('long')) / 60.0))
)
device_day = (
    exploded.groupBy('fleet', 'device_id', 'day')
    .agg(F.sum('minutes_in_day').alias('oos_minutes'))
    .persist(StorageLevel.DISK_ONLY)
)
ceiling_breaches = device_day.where(F.col('oos_minutes') > 1440.5).count()
print(f'device-days exceeding 1440 minutes after the union: {ceiling_breaches:,}  (must be 0)')
print()

restated = (
    device_day.groupBy('fleet').agg(
        F.countDistinct('device_id').alias('devices'),
        F.count('*').alias('device_days_with_oos'),
        F.sum('oos_minutes').alias('union_oos_minutes'),
        F.expr('percentile_approx(oos_minutes, 0.5)').alias('p50_min_per_affected_device_day'),
        F.expr('percentile_approx(oos_minutes, 0.9)').alias('p90_min_per_affected_device_day'),
    ).toPandas()
)
restated = restated.merge(union_by_fleet[['fleet', 'summed_minutes', 'union_minutes']], on='fleet')
_cap_min = restated['devices'] * WIN_DAYS * DAY
restated['capacity_device_hours'] = (_cap_min / 60.0).round(1)
restated['old_share_pct'] = (100.0 * restated['summed_minutes'] / _cap_min).round(2)
restated['restated_share_pct'] = (100.0 * restated['union_oos_minutes'] / _cap_min).round(2)
restated['overstatement_factor'] = (
    restated['old_share_pct'] / restated['restated_share_pct'].clip(lower=0.001)).round(2)
restated['pct_device_days_affected'] = (
    100.0 * restated['device_days_with_oos'] / (restated['devices'] * WIN_DAYS)).round(2)
cols = ['fleet', 'devices', 'capacity_device_hours', 'old_share_pct', 'restated_share_pct',
        'overstatement_factor', 'pct_device_days_affected',
        'p50_min_per_affected_device_day', 'p90_min_per_affected_device_day']
print(restated[cols].sort_values('fleet').to_string(index=False))
print()

print('=' * 78)
print('HOW TO READ THIS')
print('=' * 78)
print(
    'A.0 is new. It shows how far outage_end runs past the window and whether this source\n'
    '    carries sentinel dates. Intervals are clamped to the window before anything else,\n'
    '    which is both safer and more correct: only downtime inside the window counts.\n'
    'A.1 decides what duration_min even is. If it agrees with the clamped span, the sum and\n'
    '    the union are comparable. If duration_min is systematically smaller, it is already\n'
    '    a clipped quantity and the union is the only honest measure.\n'
    'B.1 is the decisive line. overlap_factor is how many times the average out-of-service\n'
    '    minute is counted. 1.0 means no overlap and the current metric is sound. Well above\n'
    '    1.0 means PS2 is summing concurrent episodes and the published share is inflated by\n'
    '    exactly that factor.\n'
    'C.2 says which fix applies:\n'
    '    pct_blocks_multi_subsystem high   -> legitimate multi-component recording; the fix\n'
    '                                         is the union, in the PS2 notebook, nothing else\n'
    '    pct_blocks_repeated_event_id high -> the same episode appears more than once; that\n'
    '                                         is a source or ingestion defect and belongs in\n'
    '                                         a conversation with Cubic, not a notebook patch\n'
    'E.1 is what the dashboard should say. The ceiling check must print 0.\n'
)
print(
    'WHAT THIS STILL DOES NOT TEST. Whether the underlying episodes are real at all. A union\n'
    'of spurious intervals is still spurious. It also says nothing about data after\n'
    '11 April 2026; the closure diagnostic showed boundary censoring is negligible here.\n'
)
